In [2]:
import os
import json
import numpy as np
import pandas as pd
import torch

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled.")

PyTorch: 2.14.0+cpu
CUDA available: False


In [ ]:
import glob

json_files = glob.glob(
    "/kaggle/input/**/*.json",
    recursive=True
)

print("JSON files found:", len(json_files))

for f in json_files[:20]:
    print(f)

JSON files found: 6
/kaggle/input/datasets/aditik1234/contract-nli/contract-nli/dev.json
/kaggle/input/datasets/aditik1234/contract-nli/contract-nli/train.json
/kaggle/input/datasets/aditik1234/contract-nli/contract-nli/test.json
/kaggle/input/datasets/aditik1234/legalbert/legalbert_contractnli/config.json
/kaggle/input/datasets/aditik1234/legalbert/legalbert_contractnli/tokenizer.json
/kaggle/input/datasets/aditik1234/legalbert/legalbert_contractnli/tokenizer_config.json


In [ ]:
train_path = [f for f in json_files if f.endswith("/train.json")][0]
dev_path   = [f for f in json_files if f.endswith("/dev.json")][0]
test_path  = [f for f in json_files if f.endswith("/test.json")][0]

print("Train:", train_path)
print("Dev:", dev_path)
print("Test:", test_path)

Train: /kaggle/input/datasets/aditik1234/contract-nli/contract-nli/train.json
Dev: /kaggle/input/datasets/aditik1234/contract-nli/contract-nli/dev.json
Test: /kaggle/input/datasets/aditik1234/contract-nli/contract-nli/test.json


In [ ]:
with open(train_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(dev_path, "r", encoding="utf-8") as f:
    dev_data = json.load(f)

with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Train documents:", len(train_data["documents"]))
print("Dev documents:", len(dev_data["documents"]))
print("Test documents:", len(test_data["documents"]))

Train documents: 423
Dev documents: 61
Test documents: 123


In [ ]:
label_map = {
    "NotMentioned": 0,
    "Entailment": 1,
    "Contradiction": 2
}

rows = []

for split_name, data in [
    ("train", train_data),
    ("dev", dev_data),
    ("test", test_data)
]:
    
    for doc in data["documents"]:
        
        doc_id = str(doc["id"])
        document_text = doc["text"]
        
        annotations = doc["annotation_sets"][0]["annotations"]
        
        for label_id, annotation in annotations.items():
            
            choice = annotation["choice"]
            
            rows.append({
                "document_id": doc_id,
                "split": split_name,
                "label_id": label_id,
                "choice": choice,
                "target": label_map[choice],
                "hypothesis": data["labels"][label_id]["hypothesis"]
                    if isinstance(data["labels"], dict)
                    else "",
                "document_text": document_text,
                "span_ids": annotation["spans"]
            })

ml_df = pd.DataFrame(rows)

print("Total rows:", len(ml_df))
print("\nSplit:")
print(ml_df["split"].value_counts())

print("\nLabels:")
print(ml_df["choice"].value_counts())

Total rows: 10319

Split:
split
train    7191
test     2091
dev      1037
Name: count, dtype: int64

Labels:
choice
Entailment       5017
NotMentioned     4146
Contradiction    1156
Name: count, dtype: int64


In [ ]:
print(train_data["labels"])

{'nda-11': {'short_description': 'No reverse engineering', 'hypothesis': "Receiving Party shall not reverse engineer any objects which embody Disclosing Party's Confidential Information."}, 'nda-16': {'short_description': 'Return of confidential information', 'hypothesis': 'Receiving Party shall destroy or return some Confidential Information upon the termination of Agreement.'}, 'nda-15': {'short_description': 'No licensing', 'hypothesis': 'Agreement shall not grant Receiving Party any right to Confidential Information.'}, 'nda-10': {'short_description': 'Confidentiality of Agreement', 'hypothesis': 'Receiving Party shall not disclose the fact that Agreement was agreed or negotiated.'}, 'nda-2': {'short_description': 'None-inclusion of non-technical information', 'hypothesis': 'Confidential Information shall only include technical information.'}, 'nda-1': {'short_description': 'Explicit identification', 'hypothesis': 'All Confidential Information shall be expressly identified by the D

In [ ]:
all_documents = (
    train_data["documents"]
    + dev_data["documents"]
    + test_data["documents"]
)

document_map = {
    str(doc["id"]): doc
    for doc in all_documents
}

clause_records = []

for doc in all_documents:
    
    doc_id = str(doc["id"])
    
    for span_id, (start, end) in enumerate(doc["spans"]):
        
        text = doc["text"][start:end].strip()
        
        if text:
            clause_records.append({
                "document_id": doc_id,
                "span_id": span_id,
                "text": text
            })

clauses_df = pd.DataFrame(clause_records)

print("Total clause/span records:", len(clauses_df))
print("Unique documents:", clauses_df["document_id"].nunique())

Total clause/span records: 47321
Unique documents: 607
